# Notebook 01 — ETL: Extração, Transformação e Carga

**Projeto:** FSQA Take-Home Case — Modelagem Preditiva de Default em CRE  
**Fase:** 1 de 4  
**Objetivo:** Transformar o arquivo `.csv` bruto em um `DataFrame` limpo, com tipos corretos, sem ambiguidades e pronto para análise exploratória.

---

## Visão Geral da Fase

O dataset contém **8.959 registros** de empréstimos comerciais imobiliários (CRE) com 15 variáveis cada. O arquivo bruto apresenta problemas típicos de exportação de sistemas financeiros:

- Campos monetários formatados como string com `$`, vírgulas e espaços (e.g. `" $1,460,057.06 "`)
- Percentuais como string (e.g. `"7.66%"`)
- Espaços extras em colunas categóricas (e.g. `" Fully amortizing "`)
- Datas como string no formato `MM/DD/YYYY`
- Property Class com valor `"N/A"` para imóveis não-Office
- Encoding UTF-8 com BOM (`\ufeff`)

**Output desta fase:** `df_clean` — DataFrame pandas tipado, documentado e exportado em `../data/processed/df_clean.csv`.

---
## 1. Imports e Configuração

In [40]:
# =============================================================================
# Imports
# =============================================================================
import os
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

# =============================================================================
# Caminhos
# =============================================================================
RAW_DATA_PATH    = '../data/raw/Loan default case - data set.csv'
OUTPUT_DIR       = '../data/processed/'
CLEAN_DATA_PATH  = os.path.join(OUTPUT_DIR, 'df_clean.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Ambiente configurado com sucesso.')
print(f'pandas {pd.__version__} | numpy {np.__version__}')

Ambiente configurado com sucesso.
pandas 3.0.2 | numpy 2.4.4


---
## 2. Funções de Limpeza

Seguindo o princípio de **Single Responsibility**, cada função de transformação é isolada, testável e reutilizável.

In [41]:
# =============================================================================
# Funções de Transformação — Single Responsibility
# =============================================================================

def clean_currency(series: pd.Series) -> pd.Series:
    """
    Converte campos monetários do formato string '$ x,xxx.xx' para float.

    Etapas:
        1. Remove espaços externos
        2. Remove o símbolo '$'
        3. Remove separadores de milhar ','
        4. Converte para float; valores não-numéricos tornam-se NaN

    Args:
        series: pd.Series com valores do tipo '" $1,460,057.06 "'

    Returns:
        pd.Series de float64
    """
    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(r'[\$,]', '', regex=True)
        .pipe(pd.to_numeric, errors='coerce')
    )


def clean_percent(series: pd.Series) -> pd.Series:
    """
    Converte campos percentuais do formato string 'xx.xx%' para float decimal.

    Etapas:
        1. Remove espaços externos
        2. Remove o símbolo '%'
        3. Converte para float e divide por 100

    Args:
        series: pd.Series com valores do tipo '"7.66%"'

    Returns:
        pd.Series de float64 representando o decimal (e.g. 0.0766)
    """
    return (
        series
        .astype(str)
        .str.strip()
        .str.replace('%', '', regex=False)
        .pipe(pd.to_numeric, errors='coerce')
        .div(100)
    )


def clean_categorical(series: pd.Series) -> pd.Series:
    """
    Limpa colunas categóricas: remove espaços externos e converte para Categorical.

    Args:
        series: pd.Series com valores do tipo '" Fully amortizing "'

    Returns:
        pd.Series do dtype Categorical
    """
    return series.astype(str).str.strip().astype('category')


def normalize_property_class(series: pd.Series) -> pd.Series:
    """
    Trata a coluna Property Class:
        - Valores 'N/A' (imóveis não-Office) são convertidos para np.nan
        - Demais valores são limpos de espaços externos

    Args:
        series: pd.Series com valores 'Class A', 'Class B', 'Class C' ou 'N/A'

    Returns:
        pd.Series com NaN onde aplicável
    """
    cleaned = series.astype(str).str.strip()
    cleaned = cleaned.replace('N/A', np.nan)
    return cleaned


print('Funções de limpeza definidas.')

Funções de limpeza definidas.


---
## 3. Extração — Leitura do Arquivo Bruto

In [42]:
# =============================================================================
# Leitura do CSV bruto
# O arquivo usa encoding UTF-8 com BOM (\ufeff) — usar 'utf-8-sig' para
# remover automaticamente o BOM e evitar caracteres espúrios na primeira coluna.
# dtype=str: lê TUDO como string inicialmente para controlar cada conversão
# manualmente e evitar inferências incorretas do pandas.
# =============================================================================

df_raw = pd.read_csv(
    RAW_DATA_PATH,
    encoding='utf-8-sig',
    dtype=str
)

print(f'Shape bruto: {df_raw.shape}')
print(f'Colunas: {list(df_raw.columns)}')
print()
df_raw.head(3)

Shape bruto: (8959, 15)
Colunas: ['Facility ID', 'Property type', 'Rating snapshot date', 'Original Loan Amount', 'Principal Repayment Type', 'Current Loan Balance', 'Interest rate', 'Property value', 'Net operating income', 'Contractual term', 'Months to maturity', 'Annual tenant turnover', 'Region', 'Property Class', 'Default flag']



,Facility ID,Property type,Rating snapshot date,Original Loan Amount,Principal Repayment Type,Current Loan Balance,Interest rate,Property value,Net operating income,Contractual term,Months to maturity,Annual tenant turnover,Region,Property Class,Default flag
0,ABC1000001,Office building,1/1/2020,"$1,460,057.06",Fully amortizing,"$1,460,057.06",7.66%,"$2,212,207.66","$185,161.78",120,120,39%,West,Class B,0
1,ABC1000002,Multifamily residential,1/1/2017,"$771,057.52",Partially amortizing,"$583,433.53",7.57%,"$1,041,969.62","$49,493.56",120,47,39%,Northeast,NaN,0
2,ABC1000003,Retail space,1/1/2021,"$611,478.93",Fully amortizing,"$66,243.55",7.74%,"$899,233.72","$70,679.77",120,13,21%,Northeast,NaN,0


---
## 4. Transformação — Tipagem e Limpeza

In [43]:
# =============================================================================
# Construção do DataFrame limpo
# Cada coluna é transformada de forma explícita e documentada.
# Usamos .copy() para não modificar o DataFrame bruto.
# =============================================================================

df_clean = df_raw.copy()

# --- Identificador -----------------------------------------------------------
# Facility ID: manter como string; será usado como índice ou referência
df_clean['Facility ID'] = df_clean['Facility ID'].str.strip()

# --- Variáveis Categóricas ---------------------------------------------------
df_clean['Property type']           = clean_categorical(df_clean['Property type'])
df_clean['Principal Repayment Type']= clean_categorical(df_clean['Principal Repayment Type'])
df_clean['Region']                  = clean_categorical(df_clean['Region'])

# Property Class: 'N/A' → NaN (apenas Office buildings têm classificação)
df_clean['Property Class'] = normalize_property_class(df_clean['Property Class'])

# --- Data --------------------------------------------------------------------
# Rating snapshot date: formato MM/DD/YYYY
df_clean['Rating snapshot date'] = pd.to_datetime(
    df_clean['Rating snapshot date'], format='%m/%d/%Y'
)

# --- Valores Monetários ($) --------------------------------------------------
currency_cols = [
    'Original Loan Amount',
    'Current Loan Balance',
    'Property value',
    'Net operating income',
]
for col in currency_cols:
    df_clean[col] = clean_currency(df_clean[col])

# --- Percentuais (%) ---------------------------------------------------------
percent_cols = ['Interest rate', 'Annual tenant turnover']
for col in percent_cols:
    df_clean[col] = clean_percent(df_clean[col])

# --- Inteiros ----------------------------------------------------------------
df_clean['Contractual term']  = pd.to_numeric(df_clean['Contractual term'],  errors='coerce').astype('Int64')
df_clean['Months to maturity']= pd.to_numeric(df_clean['Months to maturity'],errors='coerce').astype('Int64')

# --- Variável Resposta (target) ----------------------------------------------
# Default flag: binário {0, 1} — verificar e converter para int8
df_clean['Default flag'] = pd.to_numeric(df_clean['Default flag'], errors='coerce').astype('Int8')

print('Transformações aplicadas com sucesso.')
print()
df_clean.dtypes

Transformações aplicadas com sucesso.



Facility ID                            str
Property type                     category
Rating snapshot date        datetime64[us]
Original Loan Amount               float64
Principal Repayment Type          category
Current Loan Balance               float64
Interest rate                      float64
Property value                     float64
Net operating income               float64
Contractual term                     Int64
Months to maturity                   Int64
Annual tenant turnover             float64
Region                            category
Property Class                         str
Default flag                          Int8
dtype: object

---
## 5. Verificação de Qualidade dos Dados (DQ Checks)

Checks sistemáticos para garantir integridade antes de prosseguir para a EDA.

In [44]:
# =============================================================================
# DQ Check 1: Shape e tipos
# =============================================================================
print('='*60)
print('DQ CHECK 1 — Shape e Tipos')
print('='*60)
print(f'Linhas: {df_clean.shape[0]:,}  |  Colunas: {df_clean.shape[1]}')
print()
print(df_clean.dtypes)

DQ CHECK 1 — Shape e Tipos
Linhas: 8,959  |  Colunas: 15

Facility ID                            str
Property type                     category
Rating snapshot date        datetime64[us]
Original Loan Amount               float64
Principal Repayment Type          category
Current Loan Balance               float64
Interest rate                      float64
Property value                     float64
Net operating income               float64
Contractual term                     Int64
Months to maturity                   Int64
Annual tenant turnover             float64
Region                            category
Property Class                         str
Default flag                          Int8
dtype: object


In [45]:
# =============================================================================
# DQ Check 2: Valores nulos
# =============================================================================
print('='*60)
print('DQ CHECK 2 — Valores Nulos')
print('='*60)

null_counts = df_clean.isnull().sum()
null_pct    = (null_counts / len(df_clean) * 100).round(2)

null_summary = pd.DataFrame({
    'Nulos (n)': null_counts,
    'Nulos (%)': null_pct
}).query('`Nulos (n)` > 0')

if null_summary.empty:
    print('Nenhum valor nulo encontrado (exceto Property Class, esperado para não-Office).')
else:
    print(null_summary)

# Property Class: NaN é esperado para não-Office — verificar proporção
n_office  = (df_clean['Property type'] == 'Office building').sum()
n_na_class= df_clean['Property Class'].isnull().sum()
n_non_office = len(df_clean) - n_office
print(f'\nOffice buildings: {n_office:,}')
print(f'Non-Office (Property Class = NaN esperado): {n_non_office:,}')
print(f'Property Class NaN efetivo: {n_na_class:,}')
print(f'Consistência: {"OK" if n_na_class == n_non_office else "VERIFICAR"}')

DQ CHECK 2 — Valores Nulos
                Nulos (n)  Nulos (%)
Property Class       6247    69.7300

Office buildings: 2,712
Non-Office (Property Class = NaN esperado): 6,247
Property Class NaN efetivo: 6,247
Consistência: OK


In [46]:
# =============================================================================
# DQ Check 3: Duplicatas
# =============================================================================
print('='*60)
print('DQ CHECK 3 — Duplicatas')
print('='*60)

n_dup_id  = df_clean.duplicated('Facility ID').sum()
n_dup_row = df_clean.duplicated().sum()

print(f'Facility IDs duplicados: {n_dup_id}')
print(f'Linhas completamente duplicadas: {n_dup_row}')

if n_dup_id == 0 and n_dup_row == 0:
    print('Status: OK — nenhuma duplicata encontrada.')

DQ CHECK 3 — Duplicatas
Facility IDs duplicados: 0
Linhas completamente duplicadas: 0
Status: OK — nenhuma duplicata encontrada.


In [47]:
# =============================================================================
# DQ Check 4: Valores do target (Default flag)
# =============================================================================
print('='*60)
print('DQ CHECK 4 — Distribuição do Target')
print('='*60)

target_counts = df_clean['Default flag'].value_counts().sort_index()
target_pct    = df_clean['Default flag'].value_counts(normalize=True).sort_index().mul(100).round(2)

print('Default flag  | Count   | %')
print('-'*35)
for val in target_counts.index:
    label = 'Default' if val == 1 else 'Não-Default'
    print(f'{val} ({label:<11}) | {target_counts[val]:>6,} | {target_pct[val]:>6.2f}%')

print()
# Verificar ausência de valores fora de {0, 1}
valid_vals = set(df_clean['Default flag'].dropna().unique())
print(f'Valores únicos: {valid_vals}')
print(f'Status: {"OK" if valid_vals <= {0, 1} else "ERRO — valores inesperados"}')

DQ CHECK 4 — Distribuição do Target
Default flag  | Count   | %
-----------------------------------
0 (Não-Default) |  8,296 |  92.60%
1 (Default    ) |    663 |   7.40%

Valores únicos: {np.int8(0), np.int8(1)}
Status: OK


In [48]:
# =============================================================================
# DQ Check 5: Consistência financeira
# =============================================================================
print('='*60)
print('DQ CHECK 5 — Consistência Financeira')
print('='*60)

# 5a: Months to maturity deve ser >= 0
n_neg_mtm = (df_clean['Months to maturity'] < 0).sum()
print(f'Loans com Months to Maturity < 0 (vencidos): {n_neg_mtm}')

# 5b: Months to maturity não deve exceder Contractual Term
n_mtm_gt_ct = (df_clean['Months to maturity'] > df_clean['Contractual term']).sum()
print(f'Loans com Months to Maturity > Contractual Term: {n_mtm_gt_ct}')

# 5c: Current Balance / Original Amount — verificar outliers acima de 110%
balance_ratio = df_clean['Current Loan Balance'] / df_clean['Original Loan Amount']
n_balance_high = (balance_ratio > 1.10).sum()
print(f'Loans com Balance > 110%% do Original (potencial anomalia): {n_balance_high}')
print(f'Razão Balance/Original — mediana: {balance_ratio.median():.4f} | max: {balance_ratio.max():.4f}')

# 5d: NOI negativo (economicamente plausível, mas documentar)
n_neg_noi = (df_clean['Net operating income'] < 0).sum()
print(f'Loans com NOI negativo: {n_neg_noi} (plausível; imóvel com despesas > receitas)')

# 5e: Property Value positivo
n_neg_val = (df_clean['Property value'] <= 0).sum()
print(f'Loans com Property Value <= 0: {n_neg_val}')

DQ CHECK 5 — Consistência Financeira
Loans com Months to Maturity < 0 (vencidos): 0
Loans com Months to Maturity > Contractual Term: 0
Loans com Balance > 110%% do Original (potencial anomalia): 0
Razão Balance/Original — mediana: 0.8233 | max: 1.0000
Loans com NOI negativo: 0 (plausível; imóvel com despesas > receitas)
Loans com Property Value <= 0: 0


In [49]:
# =============================================================================
# DQ Check 6: Estatísticas descritivas
# =============================================================================
print('='*60)
print('DQ CHECK 6 — Estatísticas Descritivas (variáveis numéricas)')
print('='*60)

numeric_cols = [
    'Original Loan Amount', 'Current Loan Balance', 'Interest rate',
    'Property value', 'Net operating income', 'Contractual term',
    'Months to maturity', 'Annual tenant turnover'
]

df_clean[numeric_cols].describe().round(4)

DQ CHECK 6 — Estatísticas Descritivas (variáveis numéricas)


,Original Loan Amount,Current Loan Balance,Interest rate,Property value,Net operating income,Contractual term,Months to maturity,Annual tenant turnover
count,"8,959.0000","8,959.0000","8,959.0000","8,959.0000","8,959.0000","8,959.0000","8,959.0000","8,959.0000"
mean,"5,619,102.8081","4,152,707.2120",0.0668,"8,258,039.6769","609,533.2535",98.6360,50.0233,0.1524
std,"7,116,575.3306","5,737,360.4631",0.0082,"10,355,081.5237","844,778.4872",39.1409,36.0762,0.1204
min,"66,072.5600","2,087.8800",0.0484,"115,916.7700","1,808.8300",60.0000,1.0000,0.0100
25%,"1,723,217.3950","1,106,273.0150",0.0600,"2,570,329.0550","169,398.1000",60.0000,22.0000,0.0600
50%,"3,384,353.2900","2,361,835.5900",0.0676,"4,987,279.1400","351,252.4100",120.0000,43.0000,0.1200
75%,"6,700,877.9650","4,896,930.6500",0.0736,"9,926,114.0950","723,283.4150",120.0000,70.0000,0.2400
max,"106,146,274.8800","102,535,904.5600",0.0902,"147,698,966.2400","17,576,176.9800",180.0000,180.0000,0.5500


In [50]:
# =============================================================================
# DQ Check 7: Distribuição de variáveis categóricas
# =============================================================================
print('='*60)
print('DQ CHECK 7 — Distribuição Categórica')
print('='*60)

cat_cols = ['Property type', 'Principal Repayment Type', 'Region', 'Property Class']
for col in cat_cols:
    print(f'\n--- {col} ---')
    vc = df_clean[col].value_counts(dropna=False)
    pct = df_clean[col].value_counts(dropna=False, normalize=True).mul(100).round(2)
    result = pd.DataFrame({'Count': vc, '%': pct})
    print(result.to_string())

DQ CHECK 7 — Distribuição Categórica

--- Property type ---
                         Count       %
Property type                         
Multifamily residential   3143 35.0800
Retail space              3104 34.6500
Office building           2712 30.2700

--- Principal Repayment Type ---
                          Count       %
Principal Repayment Type               
Partially amortizing       6268 69.9600
Fully amortizing           2691 30.0400

--- Region ---
           Count       %
Region                  
Northeast   4470 49.8900
South       2268 25.3200
Midwest     1762 19.6700
West         459  5.1200

--- Property Class ---
                Count       %
Property Class               
NaN              6247 69.7300
Class A           928 10.3600
Class C           917 10.2400
Class B           867  9.6800


---
## 6. Snapshot do DataFrame Limpo

In [51]:
# =============================================================================
# Visão final do DataFrame limpo
# =============================================================================
print('Schema final do df_clean:')
print('='*60)
print(f'Linhas: {df_clean.shape[0]:,} | Colunas: {df_clean.shape[1]}')
print()

schema_df = pd.DataFrame({
    'Coluna': df_clean.columns,
    'Dtype': df_clean.dtypes.values.astype(str),
    'Nulos': df_clean.isnull().sum().values,
    'Exemplo': [str(df_clean[c].iloc[0]) for c in df_clean.columns]
})
print(schema_df.to_string(index=False))

print()
df_clean.head()

Schema final do df_clean:
Linhas: 8,959 | Colunas: 15

                  Coluna          Dtype  Nulos             Exemplo
             Facility ID            str      0          ABC1000001
           Property type       category      0     Office building
    Rating snapshot date datetime64[us]      0 2020-01-01 00:00:00
    Original Loan Amount        float64      0          1460057.06
Principal Repayment Type       category      0    Fully amortizing
    Current Loan Balance        float64      0          1460057.06
           Interest rate        float64      0              0.0766
          Property value        float64      0          2212207.66
    Net operating income        float64      0           185161.78
        Contractual term          Int64      0                 120
      Months to maturity          Int64      0                 120
  Annual tenant turnover        float64      0                0.39
                  Region       category      0                West
       

,Facility ID,Property type,Rating snapshot date,Original Loan Amount,Principal Repayment Type,Current Loan Balance,Interest rate,Property value,Net operating income,Contractual term,Months to maturity,Annual tenant turnover,Region,Property Class,Default flag
0,ABC1000001,Office building,2020-01-01,"1,460,057.0600",Fully amortizing,"1,460,057.0600",0.0766,"2,212,207.6600","185,161.7800",120,120,0.3900,West,Class B,0
1,ABC1000002,Multifamily residential,2017-01-01,"771,057.5200",Partially amortizing,"583,433.5300",0.0757,"1,041,969.6200","49,493.5600",120,47,0.3900,Northeast,NaN,0
2,ABC1000003,Retail space,2021-01-01,"611,478.9300",Fully amortizing,"66,243.5500",0.0774,"899,233.7200","70,679.7700",120,13,0.2100,Northeast,NaN,0
3,ABC1000004,Retail space,2021-01-01,"1,481,909.0600",Partially amortizing,"1,353,476.9400",0.0498,"2,117,012.9400","112,413.3900",60,34,0.2900,South,NaN,0
4,ABC1000005,Multifamily residential,2019-01-01,"4,968,492.9200",Partially amortizing,"4,140,410.7700",0.0593,"7,306,607.2400","594,027.1700",60,10,0.0900,Midwest,NaN,0


---
## 7. Exportação

In [52]:
# =============================================================================
# Exportar o DataFrame limpo
# Usar index=False para não persistir o índice numérico padrão do pandas.
# O arquivo será consumido pelo Notebook 02 (EDA) e 03 (Feature Engineering).
# =============================================================================

df_clean.to_csv(CLEAN_DATA_PATH, index=False, encoding='utf-8')

print(f'df_clean exportado com sucesso.')
print(f'Caminho: {CLEAN_DATA_PATH}')
print(f'Shape: {df_clean.shape}')

df_clean exportado com sucesso.
Caminho: ../data/processed/df_clean.csv
Shape: (8959, 15)


---
## 8. Sumário da Fase 1

| Item | Status | Detalhe |
|------|--------|---------|
| Leitura do CSV | ✅ | 8.959 linhas × 15 colunas, encoding utf-8-sig |
| Campos monetários | ✅ | `clean_currency()` aplicada em 4 colunas |
| Campos percentuais | ✅ | `clean_percent()` aplicada em 2 colunas |
| Variáveis categóricas | ✅ | `strip()` + dtype Categorical |
| Property Class N/A | ✅ | Convertido para `np.nan` |
| Datas | ✅ | `datetime64` com formato MM/DD/YYYY |
| Duplicatas | ✅ | Nenhuma encontrada |
| Target (`Default flag`) | ✅ | Binário {0, 1}, taxa de ~7.4% |
| Consistência financeira | ✅ | Documentada nos DQ Checks |
| Exportação | ✅ | `../data/processed/df_clean.csv` |

**Próxima fase:** `02_EDA.ipynb` — Análise Exploratória de Dados.